[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/6-skills.ipynb)

## Colab setup

Run the next cell first on Google Colab. It sparse-checkouts this public repo and installs packages only when the kernel is Colab. Local Jupyter and VS Code skip that work.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) or in an environment cell: `os.environ["OPENAI_API_KEY"] = "sk-..."`. Do that before the lesson setup cell. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

In [ ]:
# Colab setup. Run this cell before the other code cells.
# A fresh Colab runtime cannot import tutorial.common yet, so this cell
# sparse-checkouts the repo first. Local Jupyter and VS Code skip that.
#
# An API key is required. On Google Colab, set OPENAI_API_KEY in one of these ways:
#   * Secrets (the key icon): a secret named OPENAI_API_KEY
#   * an environment cell: os.environ["OPENAI_API_KEY"] = "sk-..."
# Locally, put the key in the repository-root .env.
# Do not commit a key. This cell does not print the value.
# Edits you make in Colab stay in the session. They do not push to GitHub.

import sys
from pathlib import Path


def _on_colab():
    try:
        import google.colab
    except ImportError:
        return False
    return Path("/content").is_dir() and google.colab is not None


def _use(root):
    if not (Path(root) / "tutorial" / "common" / "colab.py").is_file():
        return False
    text = str(root)
    if text not in sys.path:
        sys.path.insert(0, text)
    return True


ready = False
if _on_colab():
    repo = Path("/content/book-agents")
    if not _use(repo):
        import shutil
        import subprocess

        if repo.exists() and not (repo / ".git").exists():
            raise RuntimeError(
                str(repo) + " exists but is not a git checkout. "
                "Move that folder aside and run this cell again."
            )
        if repo.exists():
            shutil.rmtree(repo)
        url = "https://github.com/junlinghu/book-agents.git"
        try:
            subprocess.check_call([
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                url, str(repo),
            ])
            subprocess.check_call([
                "git", "-C", str(repo), "sparse-checkout", "set", "tutorial",
            ])
        except subprocess.CalledProcessError:
            if repo.exists():
                shutil.rmtree(repo)
            subprocess.check_call(["git", "clone", "--depth", "1", url, str(repo)])
        if not _use(repo):
            raise RuntimeError(
                "Colab setup could not find tutorial/common/colab.py after cloning."
            )
    ready = True
else:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if _use(candidate):
            ready = True
            break
    if not ready:
        print("Not Colab. Skipped clone and pip install.")

if ready:
    from tutorial.common.colab import setup_colab

    setup_colab()


# 6. Skills

A system prompt that contains every procedure gets long and stale. A skill is a procedure you load when the question matches.

`load_skill` reads `data/skills/recommend.md`. The run then searches Priya's memory and reads `faq.md`. The skill tells the agent those steps. It does not list a price. Prices and allergens stay in the FAQ. The loop, memory, and the FAQ tool are imported.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, the setup cell above reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `6-skills.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '6-skills'
LESSON_NUMBER = 6
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.memory import memory_set, reset_memory
from tutorial.common.shelf import read_shop_file  # registers the FAQ tool


## Tutorial 6 — skills

A skill is a procedure. It is not the system prompt and it is not a second menu.

In [ ]:
import re

from tutorial.common.harness import DATA, register

SKILLS = DATA / "skills"


def load_skill(args):
    """Return a procedure. The skill is not the system prompt and it is not a tool implementation."""
    name = str(args.get("name", "")).strip().lower()
    if not re.fullmatch(r"[a-z0-9-]+", name):
        return "ERROR: skill name must be recommend or restock."
    path = SKILLS / (name + ".md")
    if not path.is_file():
        return "ERROR: unknown skill " + name + ". Use recommend or restock."
    return "SKILL: " + name + "\n\n" + path.read_text(encoding="utf-8")


register(
    "load_skill",
    "Load a portable procedure by name: recommend or restock. "
    "Follow its steps. Do not treat the skill file as a second copy of the FAQ.",
    {"name": {"type": "string", "description": "recommend or restock"}},
    ["name"],
    load_skill,
)


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
reset_memory()
print("previous shift wrote memory; this message list starts empty")
print(memory_set({
    "text": "Priya is allergic to almonds.",
    "scope": "customer:priya",
    "kind": "constraint",
    "source": "previous shift",
}))
result = run_agent(
    "Recommend a pastry for Priya.",
    system_text(),
)
print("ANSWER:", result["text"])
joined = "\n".join(result["tool_log"])
check(result["stopped"] == "final", "recommendation finished")
check("load_skill" in joined, "the recommend skill was loaded")
check("memory_search" in joined, "memory from tutorial 5 was searched")
check("read_shop_file" in joined, "the FAQ was read instead of trusting the skill as a menu")
check("cardamom" in result["text"].lower() and "almond" in result["text"].lower(), "the bun and the allergy are both in the answer")
check("nut-free" in result["text"].lower(), "the skill's refusal is in the answer")
